## 1) Environment Setup
Prepare local project paths and install required libraries.


In [23]:
!nvidia-smi

Wed Apr  8 16:26:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti     Off |   00000000:01:00.0 Off |                  N/A |
| 30%   23C    P8              9W /  300W |   11899MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from pathlib import Path

PROJECT_ROOT = Path(r'/home/ridwan/Documents/indonesia-ai-nlpa-summary')
ARCHIVE_PATH = Path(r'/home/ridwan/Documents/indonesia-ai-nlpa-summary/liputan6_data.tar.gz')

print(f"Project root: {PROJECT_ROOT}")
print(f"Archive path: {ARCHIVE_PATH}")

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f"Archive not found at {ARCHIVE_PATH}")


Project root: /home/ridwan/Documents/indonesia-ai-nlpa-summary
Archive path: /home/ridwan/Documents/indonesia-ai-nlpa-summary/liputan6_data.tar.gz


In [6]:
%pip install -q transformers datasets evaluate rouge_score "accelerate>=1.1.0"

Note: you may need to restart the kernel to use updated packages.


## 2) Data Extraction and Sampling
Extract raw data, refresh destination folders, and build train/dev/test samples.

In [7]:
import os
import shutil
import tarfile

temp_extract_path = PROJECT_ROOT / '_temp_liputan6_extract'

if temp_extract_path.exists():
    shutil.rmtree(temp_extract_path)
temp_extract_path.mkdir(parents=True, exist_ok=True)

print(f"Extracting canonical data from {ARCHIVE_PATH} to {temp_extract_path}...")
with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
    canonical_members = [member for member in tar.getmembers() if member.name.startswith("liputan6_data/canonical/")]
    tar.extractall(path=temp_extract_path, members=canonical_members)
print("Temporary extraction finished.")


Extracting canonical data from /home/ridwan/Documents/indonesia-ai-nlpa-summary/liputan6_data.tar.gz to /home/ridwan/Documents/indonesia-ai-nlpa-summary/_temp_liputan6_extract...
Temporary extraction finished.


In [8]:
import shutil

sampled_extract_path = PROJECT_ROOT / 'liputan6_data'

print(f"Deleting old sampled folder at {sampled_extract_path}...")
if sampled_extract_path.exists():
    shutil.rmtree(sampled_extract_path)
    print("Old folder successfully deleted.")
else:
    print("Old folder does not exist.")

(sampled_extract_path / 'canonical').mkdir(parents=True, exist_ok=True)
print(f"Created fresh directory at {sampled_extract_path / 'canonical'}")


Deleting old sampled folder at /home/ridwan/Documents/indonesia-ai-nlpa-summary/liputan6_data...
Old folder successfully deleted.
Created fresh directory at /home/ridwan/Documents/indonesia-ai-nlpa-summary/liputan6_data/canonical


In [9]:
import os
import random
import shutil

# Configuration
TOTAL_FILES_TO_SAMPLE = 25000
train_ratio = 0.70
dev_ratio = 0.10
test_ratio = 0.20

# Paths
local_base_path = temp_extract_path / 'liputan6_data' / 'canonical'
sampled_base_path = sampled_extract_path / 'canonical'

# Calculate counts
train_count = int(TOTAL_FILES_TO_SAMPLE * train_ratio)
dev_count = int(TOTAL_FILES_TO_SAMPLE * dev_ratio)
test_count = TOTAL_FILES_TO_SAMPLE - train_count - dev_count # Ensure it adds up exactly

splits = {
    'train': train_count,
    'dev': dev_count,
    'test': test_count
}

print(f"Target distribution -> Train: {train_count}, Dev: {dev_count}, Test: {test_count}\n")

for split, count in splits.items():
    src_dir = local_base_path / split
    dst_dir = sampled_base_path / split

    dst_dir.mkdir(parents=True, exist_ok=True)

    if src_dir.exists():
        all_files = [f for f in os.listdir(src_dir) if f.endswith('.json')]
        print(f"Found {len(all_files)} files in local {split} folder.")

        files_to_copy = random.sample(all_files, min(count, len(all_files)))

        print(f"Copying {len(files_to_copy)} files to local sampled {split} folder...")
        for f in files_to_copy:
            shutil.copy2(src_dir / f, dst_dir / f)
        print(f"Finished copying {split}.\n")
    else:
        print(f"Warning: Source directory {src_dir} not found!")

print(f"Sampling and copying process completed. Sampled data is available at {sampled_base_path}")


Target distribution -> Train: 17500, Dev: 2500, Test: 5000

Found 193883 files in local train folder.
Copying 17500 files to local sampled train folder...
Finished copying train.

Found 10972 files in local dev folder.
Copying 2500 files to local sampled dev folder...
Finished copying dev.

Found 10972 files in local test folder.
Copying 5000 files to local sampled test folder...
Finished copying test.

Sampling and copying process completed. Sampled data is available at /home/ridwan/Documents/indonesia-ai-nlpa-summary/liputan6_data/canonical


## 3) Data Sanity Check
Inspect one sample file to verify schema and content shape.

In [10]:
import os
import json

sample_dir = PROJECT_ROOT / 'liputan6_data' / 'canonical' / 'train'

if sample_dir.exists():
    sample_files = [f for f in os.listdir(sample_dir) if f.endswith('.json')]
    if sample_files:
        sample_file_path = sample_dir / sample_files[0]
        print(f"Reading sample file: {sample_file_path}\n")

        with open(sample_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print("JSON Keys:", list(data.keys()))
        print("\nFull JSON Content:")
        print(json.dumps(data, indent=2, ensure_ascii=False))
    else:
        print("No JSON files found in the directory.")
else:
    print(f"Directory not found: {sample_dir}")


Reading sample file: /home/ridwan/Documents/indonesia-ai-nlpa-summary/liputan6_data/canonical/train/68807.json

JSON Keys: ['id', 'url', 'clean_article', 'clean_summary', 'extractive_summary']

Full JSON Content:
{
  "id": 68807,
  "url": "https://www.liputan6.com/news/read/68807/banjir-kembali-melanda-palembang",
  "clean_article": [
    [
      "Liputan6",
      ".",
      "com",
      ",",
      "Palembang",
      ":",
      "Setelah",
      "dilanda",
      "banjir",
      "tiga",
      "pekan",
      "silam",
      ",",
      "Kota",
      "Palembang",
      ",",
      "Sumatra",
      "Selatan",
      ",",
      "kembali",
      "digenangi",
      "air",
      "akibat",
      "meluapnya",
      "Sungai",
      "Ogan",
      "dan",
      "Sungai",
      "Kedukan",
      "."
    ],
    [
      "Banjir",
      "yang",
      "terjadi",
      "Selasa",
      "(",
      "23/12",
      ")",
      "datang",
      "pada",
      "malam",
      "hari",
      "saat",
      "air",
      "sung

## 4) Load, Structure, and N-gram EDA
Load JSON files into pandas DataFrames for each split, then run n-gram exploratory analysis to pick a better `no_repeat_ngram_size`.

In [11]:
import os
import json
import pandas as pd
from tqdm.notebook import tqdm

def load_data_split(split_path):
    records = []
    if not os.path.exists(split_path):
        print(f"Path not found: {split_path}")
        return pd.DataFrame()

    files = [f for f in os.listdir(split_path) if f.endswith('.json')]
    for f in tqdm(files, desc=f"Loading {os.path.basename(split_path)}"):
        with open(os.path.join(split_path, f), 'r', encoding='utf-8') as file:
            data = json.load(file)

            article = ' '.join([' '.join(sentence) for sentence in data.get('clean_article', [])])
            summary = ' '.join([' '.join(sentence) for sentence in data.get('clean_summary', [])])

            records.append({
                'id': data.get('id'),
                'url': data.get('url'),
                'article': article,
                'summary': summary,
                'extractive_summary': data.get('extractive_summary')
            })

    return pd.DataFrame(records)

base_dir = PROJECT_ROOT / 'liputan6_data' / 'canonical'

print("Loading datasets into Pandas DataFrames...")
df_train = load_data_split(base_dir / 'train')
df_dev = load_data_split(base_dir / 'dev')
df_test = load_data_split(base_dir / 'test')

print("\nData Loading Complete!")
print(f"Train size: {len(df_train)}")
print(f"Dev size: {len(df_dev)}")
print(f"Test size: {len(df_test)}")

display(df_train.head())


Loading datasets into Pandas DataFrames...


Loading train:   0%|          | 0/17500 [00:00<?, ?it/s]

Loading dev:   0%|          | 0/2500 [00:00<?, ?it/s]

Loading test:   0%|          | 0/5000 [00:00<?, ?it/s]


Data Loading Complete!
Train size: 17500
Dev size: 2500
Test size: 5000


,id,url,article,summary,extractive_summary
0,68807,https://www.liputan6.com/news/read/68807/banji...,"Liputan6 . com , Palembang : Setelah dilanda b...",Banjir kali ini melanda sekitar 50 rumah warga...,"[3, 4]"
1,52655,https://www.liputan6.com/news/read/52655/muncu...,"Liputan6 . com , Jakarta : Agresi militer Amer...",Kongres Uni Wanita Muslim Internasional memint...,"[1, 5]"
2,287798,https://www.liputan6.com/news/read/287798/mass...,"Liputan6 . com , Sumenep : Menolak survei seis...",Menolak survei seismik oleh salah satu perusah...,"[0, 3]"
3,218456,https://www.liputan6.com/news/read/218456/dewi...,RUMAH tangga penyanyi dan presenter Dewi Sandr...,RUMAH tangga penyanyi dan presenter Dewi Sandr...,"[0, 5, 4, 1, 3, 6, 2, 7]"
4,88746,https://www.liputan6.com/news/read/88746/abdul...,"Liputan6 . com , Jakarta : Sejumlah pengamat h...",Jaksa Agung Abdul Rahman Saleh menghadapi bany...,"[4, 5]"


In [12]:
from collections import Counter

# N-gram EDA on reference summaries to guide no_repeat_ngram_size

def tokenize_simple(text):
    return [tok for tok in str(text).lower().split() if tok.strip()]

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def top_ngrams(corpus_series, n, top_k=20):
    counter = Counter()
    for txt in corpus_series:
        toks = tokenize_simple(txt)
        counter.update(ngrams(toks, n))
    return counter.most_common(top_k)

def repeated_ngram_ratio(corpus_series, n):
    repeated = 0
    valid = 0
    for txt in corpus_series:
        toks = tokenize_simple(txt)
        ng = ngrams(toks, n)
        if not ng:
            continue
        valid += 1
        c = Counter(ng)
        if any(v > 1 for v in c.values()):
            repeated += 1
    return (repeated / valid) if valid else 0.0

summary_corpus = df_train['summary'].dropna()
print('N-gram EDA on training summaries')

for n in [2, 3, 4]:
    ratio = repeated_ngram_ratio(summary_corpus, n)
    print(f'  Repeated {n}-gram in same summary: {ratio*100:.2f}%')

for n in [2, 3, 4]:
    print(f'\nTop {n}-grams in summaries:')
    for gram, freq in top_ngrams(summary_corpus, n, top_k=15):
        print(f"  {' '.join(gram):45s} {freq}")

# Heuristic: use smallest n where repeated n-gram ratio <= 5%
threshold = 0.05
recommended_no_repeat_ngram_size = 4
for n in [2, 3, 4]:
    if repeated_ngram_ratio(summary_corpus, n) <= threshold:
        recommended_no_repeat_ngram_size = n
        break

print('\nRecommended no_repeat_ngram_size (from EDA):', recommended_no_repeat_ngram_size)

N-gram EDA on training summaries
  Repeated 2-gram in same summary: 21.03%
  Repeated 3-gram in same summary: 3.59%
  Repeated 4-gram in same summary: 0.78%

Top 2-grams in summaries:
  . .                                           1014
  . mereka                                      747
  , dan                                         746
  , jawa                                        600
  ini .                                         527
  . namun                                       495
  , jakarta                                     481
  , jatim                                       428
  , jabar                                       428
  . di                                          413
  di kawasan                                    410
  ) .                                           406
  , jateng                                      400
  ini ,                                         352
  tersebut .                                    331

Top 3-grams in summaries:
  . . . 

## 5) Tokenization and Preprocessing
Convert DataFrames to Hugging Face datasets and tokenize inputs/targets.

In [13]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# Convert Pandas DataFrames to Hugging Face Datasets
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "validation": Dataset.from_pandas(df_dev),
    "test": Dataset.from_pandas(df_test)
})

# We use a BERT2BERT model tailored for Indonesian text summarization
model_checkpoint = "cahya/bert2bert-indonesian-summarization"

print(f"Loading tokenizer from {model_checkpoint}...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Set max lengths for the articles and the summaries
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = examples["article"]
    targets = examples["summary"]

    # Tokenize the input articles
    model_inputs = tokenizer(inputs, max_length=max_input_length, padding="max_length", truncation=True)

    # Tokenize the target summaries
    labels = tokenizer(targets, max_length=max_target_length, padding="max_length", truncation=True)

    # Replace the padding token id with -100 so they are ignored by the loss function
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing datasets... This might take a minute.")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
print("\nTokenization complete!")

# Preview the keys of the first tokenized train record to ensure it contains 'input_ids' and 'labels'
print("\nFeatures in our tokenized dataset:", list(tokenized_datasets["train"][0].keys()))

Loading tokenizer from cahya/bert2bert-indonesian-summarization...
Tokenizing datasets... This might take a minute.


Map:   0%|          | 0/17500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]


Tokenization complete!

Features in our tokenized dataset: ['id', 'url', 'article', 'summary', 'extractive_summary', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']


## 6) Extractive Baseline Evaluation (Comparable Protocol)
Evaluate extractive summaries using the same protocol as the comparison table:
- prediction: extractive summary reconstructed from index pointers into `clean_article`
- target: human abstractive summary (`clean_summary`)
- split/subset: 250-sample test subset (fixed seed)
- report: ROUGE-1/2/L precision, recall, and F1

In [14]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

scores = scorer.score(
    target="Liputan6 . com , Jakarta : Artis Ussy Sulistiawaty mengaku sedikit mengerem kegiatan di bulan suci Ramadan . Tujuannya demi menjaga kondisi tubuh yang belakangan rentan terhadap penyakit . \" Rada dikurangi saja , sekarang lebih fokus sama rekaman buat album ketiga , \"ujar Ussy di Hot Shot SCTV , Ahad ( 29/8 ) . Ussy saat ini berada pada kondisi 90 persen fit . Ussy yang pada saat itu ditemani sang kekasih , Andika Pratama berencana meluncurkan album terbaru setelah Lebaran . Pelantun tembang Klik itu berharap deretan lagu barunya mampu memikat hati para pecinta musik Tanah Air . Sebelumnya Ussy dan Andika berkomitmen hanya bertemu setelah berbuka puasa . Itu semua dilakukan demi menghindari hal negatif yang dapat merusak ibadah mereka [ baca : Andhika-Ussy Bicara Soal Makna Puasa ] . ( AIS ) .",
    prediction="Ussy Sulistiawaty mengaku sedikit mengerem kegiatan di bulan Ramadan untuk menjaga kondisi tubuh ."
)

print(scores)

{'rouge1': Score(precision=0.9230769230769231, recall=0.10434782608695652, fmeasure=0.18749999999999997), 'rouge2': Score(precision=0.75, recall=0.07894736842105263, fmeasure=0.14285714285714285), 'rougeL': Score(precision=0.9230769230769231, recall=0.10434782608695652, fmeasure=0.18749999999999997)}


In [15]:
import json
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

EVAL_SAMPLE_SIZE = 250
EVAL_RANDOM_STATE = 42

# Same subset protocol used by the downstream comparison section
subset = df_test.sample(n=EVAL_SAMPLE_SIZE, random_state=EVAL_RANDOM_STATE).reset_index(drop=True)

def _resolve_row_id(raw_id):
    if isinstance(raw_id, (int, np.integer)):
        return str(int(raw_id))
    if isinstance(raw_id, float) and raw_id.is_integer():
        return str(int(raw_id))
    return str(raw_id).strip()

def reconstruct_extractive_from_indices(row_id):
    json_id = _resolve_row_id(row_id)
    json_path = base_dir / 'test' / f'{json_id}.json'

    if not json_path.exists():
        return ''

    with open(json_path, 'r', encoding='utf-8') as f:
        sample = json.load(f)

    clean_article = sample.get('clean_article', [])
    extractive = sample.get('extractive_summary', [])

    if isinstance(extractive, list) and all(isinstance(i, (int, np.integer)) for i in extractive):
        sentence_texts = [
            ' '.join(sent) if isinstance(sent, list) else str(sent)
            for sent in clean_article
        ]
        picked = [sentence_texts[i] for i in extractive if 0 <= i < len(sentence_texts)]
        return ' '.join(picked).strip()

    # Fallback for non-index formats.
    if isinstance(extractive, str):
        return extractive.strip()
    if isinstance(extractive, list):
        chunks = []
        for item in extractive:
            if isinstance(item, list):
                chunks.append(' '.join(str(tok) for tok in item))
            else:
                chunks.append(str(item))
        return ' '.join(part for part in chunks if part).strip()

    return ''

def compute_rouge_extractive(row):
    pred = reconstruct_extractive_from_indices(row['id'])
    scores = scorer.score(target=row['summary'], prediction=pred)

    result = {}
    for m in ['rouge1', 'rouge2', 'rougeL']:
        result[f'{m}_precision'] = scores[m].precision
        result[f'{m}_recall'] = scores[m].recall
        result[f'{m}_fmeasure'] = scores[m].fmeasure
    return pd.Series(result)

rouge_scores = subset.apply(compute_rouge_extractive, axis=1)
df_eval = pd.concat([subset[['id', 'summary']], rouge_scores], axis=1)

cols = [
    'rouge1_precision', 'rouge1_recall', 'rouge1_fmeasure',
    'rouge2_precision', 'rouge2_recall', 'rouge2_fmeasure',
    'rougeL_precision', 'rougeL_recall', 'rougeL_fmeasure'
]

stats_decimal = df_eval[cols].agg(['min', 'max', 'mean']).round(4)
stats_percent = (stats_decimal * 100).round(2)

print(f'Comparable extractive baseline on test subset (n={EVAL_SAMPLE_SIZE}, seed={EVAL_RANDOM_STATE})')
print('\nROUGE stats (0-1 scale):')
print(stats_decimal)
print('\nROUGE stats (% scale, directly comparable to Section 11 table):')
print(stats_percent)


Comparable extractive baseline on test subset (n=250, seed=42)

ROUGE stats (0-1 scale):
      rouge1_precision  rouge1_recall  rouge1_fmeasure  rouge2_precision  \
min             0.1489         0.2800           0.1944            0.0244   
max             0.8333         1.0000           0.8511            0.7500   
mean            0.4452         0.6392           0.5173            0.2621   

      rouge2_recall  rouge2_fmeasure  rougeL_precision  rougeL_recall  \
min          0.0333           0.0282            0.0976         0.1667   
max          0.9375           0.8000            0.8000         1.0000   
mean         0.3797           0.3054            0.3568         0.5131   

      rougeL_fmeasure  
min            0.1270  
max            0.8511  
mean           0.4148  

ROUGE stats (% scale, directly comparable to Section 11 table):
      rouge1_precision  rouge1_recall  rouge1_fmeasure  rouge2_precision  \
min              14.89          28.00            19.44              2.44   


## 7) Model and Trainer Setup
Initialize the seq2seq model, metrics, and training arguments.

In [ ]:
from transformers import (
    EncoderDecoderModel,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import evaluate
import numpy as np
from typing import Any, cast

# 1. Load the model
print(f"Loading model {model_checkpoint}...")
model = EncoderDecoderModel.from_pretrained(model_checkpoint)
model.config.tie_word_embeddings = False

# Resolve n-gram blocking size from EDA (fallback to 3 if EDA cell is skipped)
NO_REPEAT_NGRAM_SIZE = int(globals().get("recommended_no_repeat_ngram_size", 3))
print(f"Using no_repeat_ngram_size={NO_REPEAT_NGRAM_SIZE} (from EDA or fallback)")

# Set model configuration for sequence-to-sequence generation
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.encoder.vocab_size

# Set generation parameters on generation_config (required for trer# min_length=50: prevents the fine-tuned model from stopping prematurely after a single
# short sentence (EOS is predicted aggressively after "." or "," post fine-tuning).
model.generation_config.max_length = 128
model.generation_config.min_length = 50
model.generation_config.no_repeat_ngram_size = NO_REPEAT_NGRAM_SIZE
model.generation_config.early_stopping = True
model.generation_config.length_penalty = 3.0
model.generation_config.num_beams = 4
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

# 2. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 3. Metric (ROUGE)
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rouge expects a newline after each sentence
    decoded_preds = ["\n".join(pred.strip().split()) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split()) for label in decoded_labels]

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # Extract a few results
    result = {key: value * 100 for key, value in result.items()}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# 4. Training Arguments (anti-overfitting defaults)
batch_size = 32

args = Seq2SeqTrainingArguments(
    output_dir=str(PROJECT_ROOT / "bert2bert-indonesian-summarization-finetuned"),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    warmup_steps=500,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=8,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,
    fp16=True, # Enable mixed precision for faster training on GPU
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=cast(Any, tokenized_datasets["train"]),
    eval_dataset=cast(Any, tokenized_datasets["validation"]),
    data_collator=data_collator,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Trainer setup complete! Overfitting controls enabled:")
print("- Early stopping patience: 2 eval epochs")
print("- Best model auto-loaded by eval_rougeL")
print("- Warmup steps: 1000")
print("- Max epochs: 8")


Loading model cahya/bert2bert-indonesian-summarization...


Loading weights:   0%|          | 0/523 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.bert.embeddings.word_embeddings.weight to decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie decoder.cls.predictions.bias to decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
EncoderDecoderModel LOAD REPORT from: cahya/bert2bert-indonesian-summarization
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
encoder.embeddings.position_ids      | UNEXPECTED |  | 
decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

Using no_repeat_ngram_size=3 (from EDA or fallback)
Trainer setup complete! Overfitting controls enabled:
- Early stopping patience: 2 eval epochs
- Best model auto-loaded by eval_rougeL
- Warmup steps: 1000
- Max epochs: 8


## 8) Pre-fine-tuning Baseline Evaluation
Generate summaries using the **pre-trained** (not yet fine-tuned) model on a 200-sample subset of the test set.
This establishes a before/after ROUGE comparison to quantify the benefit of fine-tuning.


In [ ]:
import torch
import numpy as np
from rouge_score import rouge_scorer as rs

BASELINE_SAMPLE_SIZE = 250
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Draw a reproducible 250-sample subset from the test set
baseline_sample = df_test.sample(n=BASELINE_SAMPLE_SIZE, random_state=42).reset_index(drop=True)

def generate_summary(text, mdl, tok, max_input=512, max_new=128):
    """Tokenize an article and generate an abstractive summary via beam search."""
    inputs = tok(
        text,
        max_length=max_input,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        output_ids = mdl.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new,
            # min_length=50: fine-tuning on short Liputan6 summaries causes the model to
            # predict EOS aggressively after "." or ",". Raising min_length forces the
            # model to generate at least ~35 meaningful words before stopping.
            min_length=50,
            num_beams=4,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            length_penalty=3.0,
            early_stopping=True,
            decoder_start_token_id=tok.cls_token_id,
        )
    return tok.decode(output_ids[0], skip_special_tokens=True)

print(f"Generating pre-fine-tuning summaries on {BASELINE_SAMPLE_SIZE} test samples...")
print(f"Decoding config -> no_repeat_ngram_size={NO_REPEAT_NGRAM_SIZE}, min_length=50")
pre_ft_generated = []
for _, row in baseline_sample.iterrows():
    pred = generate_summary(row["article"], model, tokenizer)
    pre_ft_generated.append(pred)
print(f"Done. Generated {len(pre_ft_generated)} summaries.")

# Compute ROUGE on generated summaries vs. reference human summaries
scorer_eval = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

agg_pre = {
    "rouge1": {"precision": [], "recall": [], "fmeasure": []},
    "rouge2": {"precision": [], "recall": [], "fmeasure": []},
    "rougeL": {"precision": [], "recall": [], "fmeasure": []}
}

for pred, ref in zip(pre_ft_generated, baseline_sample["summary"].tolist()):
    s = scorer_eval.score(target=ref, prediction=pred)
    for k in agg_pre:
        agg_pre[k]["precision"].append(s[k].precision)
        agg_pre[k]["recall"].append(s[k].recall)
        agg_pre[k]["fmeasure"].append(s[k].fmeasure)

pre_ft_scores_detailed = {
    rouge_name: {
        metric_name: round(float(np.mean(metric_values)) * 100, 4)
        for metric_name, metric_values in metrics_dict.items()
    }
    for rouge_name, metrics_dict in agg_pre.items()
}

# Keep compatibility with downstream comparison cell (uses f-measure only)
pre_ft_scores = {k: v["fmeasure"] for k, v in pre_ft_scores_detailed.items()}

print(f"\n--- Pre-fine-tuning ROUGE Scores (mean %, {BASELINE_SAMPLE_SIZE} test samples) ---")
for rouge_name, metrics_dict in pre_ft_scores_detailed.items():
    print(
        f"  {rouge_name:6s} -> "
        f"precision: {metrics_dict['precision']:.4f}, "
        f"recall: {metrics_dict['recall']:.4f}, "
        f"fmeasure: {metrics_dict['fmeasure']:.4f}"
    )


Generating pre-fine-tuning summaries on 250 test samples...
Decoding config -> no_repeat_ngram_size=3
Done. Generated 250 summaries.

--- Pre-fine-tuning ROUGE Scores (mean %, 250 test samples) ---
  rouge1 -> precision: 29.7233, recall: 26.4975, fmeasure: 27.6765
  rouge2 -> precision: 13.1902, recall: 11.7143, fmeasure: 12.2484
  rougeL -> precision: 24.3704, recall: 21.7791, fmeasure: 22.7197


## 9) Fine-tuning
Fine-tune the BERT2BERT model on the 17,500-sample Indonesian training split.
After training the model is saved inside the project so it can be reused in future sessions.


In [18]:
import os

print("Starting fine-tuning...")
train_result = trainer.train()

print("\n--- Training Metrics ---")
print(train_result.metrics)

finetuned_save_path = PROJECT_ROOT / 'bert2bert-indonesian-finetuned'
os.makedirs(finetuned_save_path, exist_ok=True)
trainer.save_model(finetuned_save_path)
tokenizer.save_pretrained(finetuned_save_path)
print(f"\nFine-tuned model saved to: {finetuned_save_path}")


Starting fine-tuning...


/home/ridwan/anaconda3/envs/pioanaconda/lib/python3.11/site-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:445: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,1.415850,2.191872,31.991600,14.720000,27.714000,31.965500,33.488800
2,1.031999,2.230658,32.117300,14.510900,27.611300,32.103100,33.839200
3,0.836006,2.317508,30.970100,13.709100,26.656000,30.986100,33.748400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/ridwan/anaconda3/envs/pioanaconda/lib/python3.11/site-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:445: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embeddings.LayerNorm.weight', 'encoder.embeddings.LayerNorm.bias', 'encoder.encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.encoder.layer.0.output.LayerNorm.weight', 'encoder.encoder.layer.0.output.LayerNorm.bias', 'encoder.encoder.layer.1.attention.output.LayerNorm.weight', 'encoder.encoder.layer.1.attention.output.LayerNorm.bias', 'encoder.encoder.layer.1.output.LayerNorm.weight', 'encoder.encoder.layer.1.output.LayerNorm.bias', 'encoder.encoder.layer.2.attention.output.LayerNorm.weight', 'encoder.encoder.layer.2.attention.output.LayerNorm.bias', 'encoder.encoder.layer.2.output.LayerNorm.weight', 'encoder.encoder.layer.2.output.LayerNorm.bias', 'encoder.encoder.layer.3.attention.output.LayerNorm.weight', 'encoder.encoder.layer.3.attention.output.LayerNorm.bias', 'encoder.encoder.layer.3.output.LayerNorm.weight', 'encoder.encoder.layer.3.output.Laye


--- Training Metrics ---
{'train_runtime': 1002.0738, 'train_samples_per_second': 139.71, 'train_steps_per_second': 8.734, 'total_flos': 3.617535688704e+16, 'train_loss': 1.0946184212953707, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuned model saved to: /home/ridwan/Documents/indonesia-ai-nlpa-summary/bert2bert-indonesian-finetuned


## 10) Post-fine-tuning Evaluation
Evaluate the fine-tuned model in two ways:

1. **Trainer evaluate** — official ROUGE on the full 5,000-sample tokenized test split (uses `compute_metrics` with `predict_with_generate=True`).
2. **Direct inference ROUGE** — run `model.generate()` on the **same 200-sample baseline subset** used in Section 8, so results are directly comparable.


In [19]:
# 10a. Official Trainer evaluation on the full tokenized test split
print("Running evaluation on the full tokenized test split (5,000 samples)...")
eval_results = trainer.evaluate(tokenized_datasets["test"])
print("\n--- Official Test-set ROUGE (full 5,000 samples) ---")
for k, v in eval_results.items():
    print(f"  {k}: {v}")


Running evaluation on the full tokenized test split (5,000 samples)...


/home/ridwan/anaconda3/envs/pioanaconda/lib/python3.11/site-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:445: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)



--- Official Test-set ROUGE (full 5,000 samples) ---
  eval_loss: 2.0063400268554688
  eval_rouge1: 34.1283
  eval_rouge2: 16.6807
  eval_rougeL: 29.5135
  eval_rougeLsum: 34.1315
  eval_gen_len: 33.219
  eval_runtime: 203.944
  eval_samples_per_second: 24.517
  eval_steps_per_second: 1.535
  epoch: 3.0


In [20]:
# 10b. Direct inference ROUGE on the same 250-sample baseline subset
print("Generating post-fine-tuning summaries on the same 250 baseline test samples...")
post_ft_generated = []
model.eval()
for _, row in baseline_sample.iterrows():
    pred = generate_summary(row["article"], model, tokenizer)
    post_ft_generated.append(pred)
print(f"Done. Generated {len(post_ft_generated)} summaries.")

agg_post = {
    "rouge1": {"precision": [], "recall": [], "fmeasure": []},
    "rouge2": {"precision": [], "recall": [], "fmeasure": []},
    "rougeL": {"precision": [], "recall": [], "fmeasure": []}
}

for pred, ref in zip(post_ft_generated, baseline_sample["summary"].tolist()):
    s = scorer_eval.score(target=ref, prediction=pred)
    for k in agg_post:
        agg_post[k]["precision"].append(s[k].precision)
        agg_post[k]["recall"].append(s[k].recall)
        agg_post[k]["fmeasure"].append(s[k].fmeasure)

post_ft_scores_detailed = {
    rouge_name: {
        metric_name: round(float(np.mean(metric_values)) * 100, 4)
        for metric_name, metric_values in metrics_dict.items()
    }
    for rouge_name, metrics_dict in agg_post.items()
}

# Keep compatibility with downstream comparison cell (uses f-measure only)
post_ft_scores = {k: v["fmeasure"] for k, v in post_ft_scores_detailed.items()}

print("\n--- Post-fine-tuning ROUGE Scores (mean %, 250 test samples) ---")
for rouge_name, metrics_dict in post_ft_scores_detailed.items():
    print(
        f"  {rouge_name:6s} -> "
        f"precision: {metrics_dict['precision']:.4f}, "
        f"recall: {metrics_dict['recall']:.4f}, "
        f"fmeasure: {metrics_dict['fmeasure']:.4f}"
    )


Generating post-fine-tuning summaries on the same 250 baseline test samples...
Done. Generated 250 summaries.

--- Post-fine-tuning ROUGE Scores (mean %, 250 test samples) ---
  rouge1 -> precision: 35.1133, recall: 27.3436, fmeasure: 30.2847
  rouge2 -> precision: 15.8417, recall: 12.3842, fmeasure: 13.7021
  rougeL -> precision: 30.5339, recall: 23.7488, fmeasure: 26.3222


## 11) Comparison Table & Qualitative Demo

### 11a) ROUGE Comparison Table
Compare our model (before and after fine-tuning) against the paper's published baselines on the Liputan6 canonical test set.

### 11b) Qualitative Examples
Inspect 5 sample articles side-by-side: reference summary, pre-fine-tuned output, and post-fine-tuned output.


In [21]:
import json
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer as rs

# Paper-reported baselines on the canonical test set (Table 3, AACL 2020 paper)
paper_baselines = {
    "Lead-2 (paper)":              {"rouge1": 36.68, "rouge2": 20.23, "rougeL": 33.71},
    "PTGen (paper)":               {"rouge1": 36.10, "rouge2": 19.19, "rougeL": 33.56},
    "BertAbs mBERT (paper)":       {"rouge1": 39.48, "rouge2": 21.59, "rougeL": 36.72},
    "BertExtAbs mBERT (paper)":    {"rouge1": 39.81, "rouge2": 21.84, "rougeL": 37.02},
    "BertAbs IndoBERT (paper)":    {"rouge1": 40.94, "rouge2": 23.01, "rougeL": 37.89},
    "BertExtAbs IndoBERT (paper) ★": {"rouge1": 41.08, "rouge2": 22.85, "rougeL": 38.01},
}

def normalize_extractive_summary(value):
    """Normalize non-index extractive_summary values into plain text."""
    if isinstance(value, str):
        return value.strip()

    if isinstance(value, list):
        chunks = []
        for item in value:
            if isinstance(item, list):
                chunks.append(" ".join(str(tok) for tok in item))
            else:
                chunks.append(str(item))
        return " ".join(part for part in chunks if part).strip()

    if pd.isna(value):
        return ""

    return str(value).strip()

def _resolve_row_id(raw_id):
    """Convert id values from dataframe into canonical filename stem."""
    if isinstance(raw_id, (int, np.integer)):
        return str(int(raw_id))
    if isinstance(raw_id, float) and raw_id.is_integer():
        return str(int(raw_id))
    return str(raw_id).strip()

def reconstruct_extractive_from_indices(row_id):
    """Rebuild extractive summary text from clean_article sentence indices in test JSON."""
    json_id = _resolve_row_id(row_id)
    json_path = base_dir / "test" / f"{json_id}.json"

    if not json_path.exists():
        return ""

    with open(json_path, "r", encoding="utf-8") as f:
        sample = json.load(f)

    clean_article = sample.get("clean_article", [])
    extractive = sample.get("extractive_summary", [])

    # Primary path: extractive_summary is a list of sentence indices into clean_article.
    if isinstance(extractive, list) and all(isinstance(i, (int, np.integer)) for i in extractive):
        sentence_texts = [
            " ".join(sent) if isinstance(sent, list) else str(sent)
            for sent in clean_article
        ]
        picked = [sentence_texts[i] for i in extractive if 0 <= i < len(sentence_texts)]
        return " ".join(picked).strip()

    # Fallback if a sample stores extractive content in text/list form.
    return normalize_extractive_summary(extractive)

# New baseline: dataset-provided extractive summary on the same baseline subset
scorer_cmp = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
extractive_preds = [reconstruct_extractive_from_indices(row_id) for row_id in baseline_sample["id"].tolist()]

agg_ext = {"rouge1": [], "rouge2": [], "rougeL": []}
for pred, ref in zip(extractive_preds, baseline_sample["summary"].tolist()):
    s = scorer_cmp.score(target=ref, prediction=pred)
    for k in agg_ext:
        agg_ext[k].append(s[k].fmeasure)

extractive_scores = {k: round(float(np.mean(v)) * 100, 2) for k, v in agg_ext.items()}

# Our model results on the same 250-sample baseline subset
our_results = {
    "Extractive summary (baseline subset)": {k: extractive_scores[k] for k in ["rouge1", "rouge2", "rougeL"]},
    "BERT2BERT (pre-fine-tuning)":         {k: pre_ft_scores[k] for k in ["rouge1", "rouge2", "rougeL"]},
    "BERT2BERT (fine-tuned ours)":         {k: post_ft_scores[k] for k in ["rouge1", "rouge2", "rougeL"]},
}

all_results = {**paper_baselines, **our_results}
comparison_df = pd.DataFrame(all_results).T.rename(columns={
    "rouge1": "ROUGE-1", "rouge2": "ROUGE-2", "rougeL": "ROUGE-L"
})
comparison_df = comparison_df.round(2)

print("=" * 60)
print("ROUGE Comparison — Liputan6 Canonical Test Set")
print("(Paper baselines: full ~10,972 test samples;")
print(" Our model + extractive baseline: same 250 sampled test articles)")
print("=" * 60)
display(comparison_df)


ROUGE Comparison — Liputan6 Canonical Test Set
(Paper baselines: full ~10,972 test samples;
 Our model + extractive baseline: same 250 sampled test articles)


,ROUGE-1,ROUGE-2,ROUGE-L
Lead-2 (paper),36.68,20.23,33.71
PTGen (paper),36.10,19.19,33.56
BertAbs mBERT (paper),39.48,21.59,36.72
BertExtAbs mBERT (paper),39.81,21.84,37.02
BertAbs IndoBERT (paper),40.94,23.01,37.89
BertExtAbs IndoBERT (paper) ★,41.08,22.85,38.01
Extractive summary (baseline subset),51.73,30.54,41.48
BERT2BERT (pre-fine-tuning),27.68,12.25,22.72
BERT2BERT (fine-tuned ours),30.28,13.70,26.32


In [24]:
DEMO_N = 10

print("=" * 80)
print(f"Qualitative Demo — {DEMO_N} Sample Summaries")
print("=" * 80)

for i in range(DEMO_N):
    row = baseline_sample.iloc[i]
    # Truncate article preview to first 300 chars to keep output readable
    article_preview = row["article"][:300].rsplit(" ", 1)[0] + " ..."

    print(f"\n[Sample {i+1}]")
    print(f"  Article (first 300 chars):\n    {article_preview}")
    print(f"\n  Reference Summary:\n    {row['summary']}")
    print(f"\n  Pre-fine-tuning Generated:\n    {pre_ft_generated[i]}")
    print(f"\n  Post-fine-tuning Generated:\n    {post_ft_generated[i]}")
    print("-" * 80)


Qualitative Demo — 10 Sample Summaries

[Sample 1]
  Article (first 300 chars):
    Liputan6 . com , Jakarta : Ratusan karyawan toko serba ada Matahari kembali berunjuk rasa di Gedung Departemen Tenaga Kerja dan Transmigrasi , Jalan Gatot Subroto , Jakarta Selatan , Senin ( 12/11 ) siang . Demonstrasi itu adalah aksi lanjutan serupa di tempat yang sama , Jumat pekan silam [ baca : ...

  Reference Summary:
    Ratusan karyawan Toserba Matahari kembali berunjuk rasa di Gedung Depnakertrans . Mereka menuntut manajemen PT Matahari Putra Prima memberikan uang makan dan transportasi .

  Pre-fine-tuning Generated:
    sejumlah karyawan toko serba ada kembali berunjuk rasa di tempat yang sama, pekan silam. mereka bertekad untuk melanjutkan aksi yang sama hingga tuntutan mereka dipenuhi.

  Post-fine-tuning Generated:
    karyawan kembali berunjuk rasa di dan,. karyawan tetap sama yakni meminta uang makan sebesar 10 ribu per hari dan ongkos transportasi.
--------------------------------------